# 🧬 Cellular Reasoning Fabric (CRF) - Industry Standard Benchmark Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasirusman85/pc-ai/blob/master/notebooks/crf_gpu_colab.ipynb)

Evaluate **Cellular Reasoning Fabric (CRF)** and **Hybrid Transformer-CRF** on industry-standard benchmarks:
- 🧮 **GSM8K** (Grade School Math Multi-Step Reasoning)
- 🐍 **HumanEval** (Python Code Generation)
- ⚡ **Dynamic Halting & FLOP Savings** (Adaptive Computation Time)

## Step 1: Environment Setup & Clone Repository

In [ ]:
# Clone repo if running in Google Colab
import os
if not os.path.exists('src/crf_reasoning'):
    !git clone https://github.com/yasirusman85/pc-ai.git
    %cd pc-ai
else:
    !git pull origin master

# Install dependencies
!pip install -r requirements.txt
!pip install datasets --quiet
!pip install -e .

## Step 2: Verify GPU Hardware Acceleration

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Device Memory (GB):", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
else:
    print("⚠️ Running on CPU! Enable GPU via Runtime -> Change runtime type -> T4 GPU in Colab.")

## Step 3: Run Industry Standard Benchmarks (GSM8K Math & HumanEval Code)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Evaluating GSM8K and HumanEval on: {device}")

!PYTHONPATH=. python scripts/eval_standard_benchmarks.py --device {device} --batch_size 16 --seq_len 128

## Step 4: Run Full Benchmark Comparison Suite

In [ ]:
!PYTHONPATH=. python scripts/benchmark.py

## Step 5: Test Dynamic Halting (ACT) & FLOP Savings

In [ ]:
import sys
sys.path.insert(0, 'src')
import torch
from crf_reasoning.crf_vectorized import CRFLanguageModel, AblationConfig

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Enable Dynamic Halting with convergence threshold
cfg = AblationConfig(use_dynamic_halting=True, halt_threshold=0.02)
model = CRFLanguageModel(
    vocab_size=1000,
    d_model=128,
    d_hidden=64,
    n_init_cells=32,
    max_cells=256,
    n_crf_steps=12,
    k_neighbors=4,
    cfg=cfg,
).to(dev)

x = torch.randint(0, 1000, (4, 32), device=dev)
logits, loss, met = model(x, targets=x, collect_metrics=True)

print(f"Device: {dev}")
print(f"Max Steps Configured: 12")
print(f"Actual Reasoning Steps Executed: {met.steps_executed}")
print(f"FLOPs Saved: {met.flop_savings_pct:.1f}%")
print(f"Active Cell Dynamics -> Splits: {met.n_splits} | Merges: {met.n_merges} | Deaths: {met.n_deaths}")